# Processing MITGCM surface data for the Spatial Scale Forcing Analysis 

**Purpose**: Code for processing surface data from the MITGCM for investigating forcing mechanisms. Here, we will generate spatial maps of $Q_{net}$ and wind stress (plus others)off the coast of Point Conception. 

**Luke Colosi | lcolosi@ucsd.edu**

Import python libraries

In [1]:
import sys
import numpy as np
import xarray as xr
from xmitgcm import open_mdsdataset

Set data analysis parameters

In [2]:
# Model parameters 
delta_t = 150  # Time steps of the raw model run (by raw, I mean the time increments that the model is ran at, not time increments that the diagnostics are output at). Units: seconds

# Set time and space parameters  
depth = 0                                                      # Specifies the depth level for the spatial map (units: meters). Options: 0, 10, 75, 1000
lat_bnds  = [33.0, 35.0]                                          # Specifies the latitude bounds for the region to analyze
lon_bnds  = [237.0, 240.0]                                        # Specifies the longitude bounds for the region to analyze
encoding  = {'time': {'units': 'seconds since 2015-12-01 2:00'}}  # Specifies the start time of the model run

# Set path to project directory
PATH_GRID   = '/data/SO2/SWOT/GRID/BIN/'                    # Space and time grid of the model 
PATH_OUTPUT = '/data/SO2/SWOT/MARA/RUN4_LY/DIAGS_DLY/'      # Diagnostics of the model
PATH_nc     = '/data/SO3/lcolosi/mitgcm/SWOT_MARA_RUN4_LY/' # Directory to save netCDFs 
file_dim    = '2D'                                          # Set the dimension of the data (to include the depth or not where 3D is T,S,drhodr,vel and 2D is etan)

Load the grid and diagnostics data into a python structure.

In [5]:
# Create dataset 
ds = open_mdsdataset(
    PATH_OUTPUT,                    # File path where the model output data is stored (.data and .meta files)
    PATH_GRID,                      # File path to the grid data of the model 
    iters='all',                    # Specifies which iteration of the data to load
    delta_t=delta_t, 
    ignore_unknown_vars=False,      # Specifies whether to ignore any unknown variables that may appear in the dataset
    prefix=['diag_' + file_dim],   # List of prefixes to filter the variables in the dataset
    ref_date="2015-01-01 02:00:00", # Specifies the starting point of the simulation time (which may include the spin up time before diagnostics are output)
    geometry='sphericalpolar'       # Specifies the  grid's geometry is spherical-polar. 
)

# Convert all variables and coordinates in the dataset to little-endian (the format how multi-byte values are stored into memory)

#--- Variables ---#
for var in ds.data_vars:
    if ds[var].dtype.byteorder == '>' or (ds[var].dtype.byteorder == '=' and sys.byteorder == "big"):  # Check if big-endian
        ds[var] = ds[var].astype(ds[var].dtype.newbyteorder('<'))

#--- Coordinates ---# 
for coord in ds.coords:
    if ds[coord].dtype.byteorder == '>'or (ds[coord].dtype.byteorder == '=' and sys.byteorder == "big"):  # Check if big-endian
        ds[coord] = ds[coord].astype(ds[coord].dtype.newbyteorder('<'))

In [32]:
print(ds['oceTAUX'])

<xarray.DataArray 'oceTAUX' (time: 1099, YC: 576, XG: 672)> Size: 2GB
dask.array<concatenate, shape=(1099, 576, 672), dtype=float32, chunksize=(1, 576, 672), chunktype=numpy.ndarray>
Coordinates:
  * YC       (YC) float32 2kB 31.01 31.03 31.05 31.07 ... 42.93 42.95 42.97
  * XG       (XG) float32 3kB 230.0 230.0 230.0 230.1 ... 243.9 243.9 244.0
    dyG      (YC, XG) float32 2MB dask.array<chunksize=(576, 672), meta=np.ndarray>
    dxC      (YC, XG) float32 2MB dask.array<chunksize=(576, 672), meta=np.ndarray>
    rAw      (YC, XG) float32 2MB dask.array<chunksize=(576, 672), meta=np.ndarray>
    maskInW  (YC, XG) bool 387kB dask.array<chunksize=(576, 672), meta=np.ndarray>
    iter     (time) int64 9kB dask.array<chunksize=(1,), meta=np.ndarray>
  * time     (time) datetime64[ns] 9kB 2015-08-20T02:00:00 ... 2018-08-22T02:...
Attributes:
    standard_name:  oceTAUX
    long_name:      zonal surface wind stress, >0 increases uVel
    units:          N/m^2
    mate:           oceTAUY


Slice array based on longitude and latitude bounds of the region

In [ ]:
# Extract scalar fields 
tflux = ds['TFLUX'].sel(YC=slice(*lat_bnds), 
                        XC=slice(*lon_bnds))
sflux  = ds['SFLUX'].sel(YC=slice(*lat_bnds), 
                         XC=slice(*lon_bnds))
oceQsw  = ds['oceQsw'].sel(YC=slice(*lat_bnds), 
                           XC=slice(*lon_bnds))
oceFWflx  = ds['oceFWflx'].sel(YC=slice(*lat_bnds), 
                               XC=slice(*lon_bnds))
oceTAUX  = ds['oceTAUX'].sel(YC=slice(*lat_bnds), 
                             XG=slice(*lon_bnds))
oceTAUY  = ds['oceTAUY'].sel(YG=slice(*lat_bnds), 
                             XC=slice(*lon_bnds))
EXFuwind  = ds['EXFuwind'].sel(YC=slice(*lat_bnds), 
                               XC=slice(*lon_bnds))
EXFvwind  = ds['EXFvwind'].sel(YC=slice(*lat_bnds), 
                               XC=slice(*lon_bnds))

KeyError: "'XG' is not a valid dimension or coordinate for Dataset with dimensions FrozenMappingWarningOnValuesAccess({'XC': 672, 'YC': 576, 'time': 1099})"

Save data in netcdf files

In [29]:
# Set the dictionary of variables to save
vars_to_save = {
    'TFLUX': tflux,
    'SFLUX': sflux,
    'oceQsw': oceQsw,
    'oceFWflx': oceFWflx,
    'oceTAUX': oceTAUX,
    'oceTAUY': oceTAUY,
    'EXFuwind': EXFuwind,
    'EXFvwind': EXFvwind
}

# Loop through each variable and save efficiently
for var_name, da in vars_to_save.items():
    
    # Chunk along time for faster write (adjust chunk size if needed)
    if 'time' in da.dims:
        da = da.chunk({'time': 1000})
    
    # Load into memory before saving (triggers computation)
    da = da.load()
    
    # Print status
    print(f"Saving {var_name}...")

    # Save to NetCDF file
    da.to_netcdf(
        f"{PATH_nc}{var_name}_CCS4_DLY_map_2D.nc",
        engine='scipy',                # Fastest NetCDF writer (writes NetCDF3)
        format='NETCDF3_64BIT',        # Simple + compatible format
        encoding=encoding            
    )


Saving TFLUX...
Saving SFLUX...
Saving oceQsw...
Saving oceFWflx...
Saving oceTAUX...
Saving oceTAUY...
Saving EXFuwind...
Saving EXFvwind...


In [30]:
PATH_nc

'/data/SO3/lcolosi/mitgcm/SWOT_MARA_RUN4_LY/'